In [ ]:
import numpy as np

_NODES, _WEIGHTS = np.polynomial.legendre.leggauss(512)
_NODES, _WEIGHTS = 0.5 * (_NODES + 1), 0.5 * _WEIGHTS   # Gauss-Legendre on [0, 1]
_U_GRID = np.geomspace(10.0, 1e5, 121)


def heston_stock_call(S, K, T, r, q, v0, kappa, theta, xi, rho):
    """European call under the Heston (1993) model, for a stock.
    Everything annual and decimal. v0, theta are VARIANCES (0.16 = 40% vol). T in years (> 0).
    K can be a number or an array. Put = call - S*exp(-q*T) + K*exp(-r*T).
    q = continuous dividend yield. Discrete dividends (QCOM): pass S minus the present value
    of dividends paid before expiry as S, and q = 0.
    Daily % vol forecast -> v0 = (vol * sqrt(252) / 100) ** 2.
    Priced by numerical integration of the Heston characteristic function (Lewis 2000;
    Albrecher et al. 2007 form, which avoids a known numerical instability)."""
    K = np.asarray(K, float)
    F, D = S * np.exp((r - q) * T), np.exp(-r * T)

    def cf(z):  # characteristic function of ln(S_T / F)
        b = kappa - 1j * rho * xi * z
        d = np.sqrt(b * b + xi * xi * (1j * z + z * z))
        g = (b - d) / (b + d)
        e = np.exp(-d * T)
        C = kappa * theta / xi ** 2 * ((b - d) * T - 2 * np.log((1 - g * e) / (1 - g)))
        Dv = (b - d) / xi ** 2 * (1 - e) / (1 - g * e)
        return np.exp(C + Dv * v0)

    # integrand decay depends strongly on T and xi, so find the cut-off instead of assuming one
    size = np.abs(cf(_U_GRID - 0.5j)) / (_U_GRID ** 2 + 0.25)
    above = np.nonzero(size >= 1e-11)[0]
    U = _U_GRID[min(above[-1] + 1, len(_U_GRID) - 1)] if above.size else _U_GRID[0]
    u, w = U * _NODES, U * _WEIGHTS
    integral = (np.real(np.exp(1j * np.outer(np.log(F / K).ravel(), u)) * cf(u - 0.5j)) / (u * u + 0.25)) @ w
    return D * (F - np.sqrt(F * K) * integral.reshape(K.shape) / np.pi)
